In [ ]:
## Clone repo và cài open_clip_torch

# Bật GPU (T4 x2 hoặc P100) và bật Internet trên Kaggle
REPO_URL = "https://github.com/tanq1965/gcd-text-leak.git"
# Nhánh chứa code N4 của Kiệt
BRANCH = "feat/n4-extract-features"
REPO = "/kaggle/working/gcd-text-leak"

!git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git branch --show-current
!pip install open_clip_torch pillow tqdm pandas
!python -c "import torch, open_clip; print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())"

In [ ]:
## Thiết lập đường dẫn dữ liệu & Symlink cho Cars

import os

CUB_ROOT = "/kaggle/input/datasets/wenewone/cub2002011"
CARS_SRC = "/kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars"
AIRCRAFT_ROOT = "/kaggle/input/datasets/seryouxblaster764/fgvc-aircraft/fgvc-aircraft-2013b/fgvc-aircraft-2013b"
CAR_ROOT = "/kaggle/working/cars/stanford_cars"

# Setup symlink cho Stanford Cars
if not os.path.exists(CAR_ROOT):
  os.makedirs(f"{CAR_ROOT}/devkit", exist_ok=True)
  for d in ["cars_train", "cars_test"]:
    if not os.path.lexists(f"{CAR_ROOT}/{d}"):
      os.symlink(f"{CARS_SRC}/{d}", f"{CAR_ROOT}/{d}")
  for f in os.listdir(f"{CARS_SRC}/devkit"):
    if not os.path.lexists(f"{CAR_ROOT}/devkit/{f}"):
      os.symlink(f"{CARS_SRC}/devkit/{f}", f"{CAR_ROOT}/devkit/{f}")
  lbl = f"{CAR_ROOT}/devkit/cars_test_annos_withlabels.mat"
  if not os.path.lexists(lbl):
    os.symlink(f"{CARS_SRC}/cars_test_annos_withlabels.mat", lbl)

print("Cấu hình đường dẫn dữ liệu: HOÀN TẤT")

In [ ]:
## Chạy trích xuất đặc trưng ảnh cho cả 3 tập

import sys

# 1. Trích xuất CUB (5.994 ảnh train)
print("=== TRÍCH XUẤT CUB ===")
!python -m src.extract_img_feats --dataset cub --split_csv splits/cub.csv --data-root {CUB_ROOT} --output_dir feats --batch_size 64

# 2. Trích xuất Stanford Cars (8.144 ảnh train)
print("\n=== TRÍCH XUẤT STANFORD CARS ===")
!python -m src.extract_img_feats --dataset scars --split_csv splits/scars.csv --data-root {CAR_ROOT} --output_dir feats --batch_size 64

# 3. Trích xuất FGVC-Aircraft (6.667 ảnh trainval)
print("\n=== TRÍCH XUẤT FGVC-AIRCRAFT ===")
!python -m src.extract_img_feats --dataset aircraft --split_csv splits/aircraft.csv --data-root {AIRCRAFT_ROOT} --output_dir feats --batch_size 64

In [ ]:
## Tự kiểm tra (Sanity Check) ma trận đặc trưng

import numpy as np
import pandas as pd

expected = {"cub": 5994, "scars": 8144, "aircraft": 6667}

print("=== KIỂM TRA CHẤT LƯỢNG ĐẶC TRƯNG N4 ===")
for ds, exp_len in expected.items():
  feat_path = f"feats/{ds}_img.npy"
  ids_path = f"feats/{ds}_img_ids.npy"
  split_path = f"splits/{ds}.csv"

  feats = np.load(feat_path)
  ids = np.load(ids_path, allow_pickle=True)
  df = pd.read_csv(split_path)

  # 1. Kiểm tra shape
  assert feats.shape == (
      exp_len,
      512,
  ), f"{ds}: Shape sai! Kỳ vọng ({exp_len}, 512), nhận được {feats.shape}"

  # 2. Kiểm tra chuẩn hóa L2 (norm vector phải xấp xỉ 1.0)
  norms = np.linalg.norm(feats, axis=1)
  assert np.allclose(norms, 1.0, atol=1e-5), f"{ds}: Vector chưa chuẩn hoá L2!"

  # 3. Kiểm tra khớp thứ tự image_id
  assert np.array_equal(
      ids.astype(str), df["image_id"].astype(str).values
  ), f"{ds}: image_id bị lệch thứ tự với splits/{ds}.csv!"

  print(
      f"✔ {ds.upper():8s}: Shape {feats.shape} chuẩn | L2-norm OK | Khớp 100%"
      " image_id"
  )

In [ ]:
## Chạy thử src.eval trên đặc trưng THẬT của CUB (Kiểm chứng pilot Dừng 1)

# Chạy thử 1 seed của bản A (chỉ dùng ảnh) trên CUB
!python -m src.eval --dataset cub --variant A --splits splits/cub.csv --feats feats/cub_img.npy --seed 0 --device cuda --out /kaggle/working/cub_A_test_s0.json

import json
rec = json.loads(open("/kaggle/working/cub_A_test_s0.json").read())
print("\n=== KẾT QUẢ TEST THỰC TẾ TRÊN CUB (BẢN A - ẢNH THẬT) ===")
print(f"ACC All: {rec.get('all', 0):.2f}% | ACC Old: {rec.get('old', 0):.2f}% | ACC New: {rec.get('new', 0):.2f}%")

In [ ]:
## Nén thư mục feats/ để tải về nộp

import shutil

zip_path = shutil.make_archive("/kaggle/working/feats_n4", "zip", "feats")
print(f"Đã nén xong: {zip_path}")
print("Các file đã nén:", os.listdir("feats"))